# `sm_rl` — a guided walkthrough

**Rediscovering the Standard Model quark sector with reinforcement learning.**

An agent builds a *model* — a gauge group plus an ordered list of matter "partons" —
one small edit at a time. A Mathematica/GroupMath kernel computes the model's
**gauge-singlet bound-state (hadron) spectrum**, and a reward measures how well that
spectrum matches the real light-hadron spectrum. The target to rediscover is
**SU(3) colour with `(u, d, s)` quarks** (antiquarks come for free from CPT).

This notebook walks the whole stack bottom-up and runs each piece:

| # | Layer | What you'll see |
|---|-------|-----------------|
| 0 | Setup | uv env, imports, engine bootstrap |
| 1 | **The Wolfram kernel** | `Engine` primitives; a line-by-line read of `spinSinglets.m` / `GetSpins` |
| 2 | **Group ladder** | groups as a graph, reps as dimension-ordered indices, clamping |
| 3 | **State & actions** | `Model`/`Parton`, the flat maskable action space |
| 4 | **Spectrum** | CPT expansion, operator enumeration, the canonical key |
| 5 | **The cache** | why every miss costs a Mathematica call, and how it's shared |
| 6 | **The target** | building `(u,d,s)` and the `[EM,B,S,I3]` charge convention |
| 7 | **Reward metrics** | count, the *broken* hungarian, and the f1 fix — with the offline landscape analysis |
| 8 | **Tokenizer** | featurizing a model into tensors (group-invariant rep scalars) |
| 9 | **Transformer** | the actor-critic over `[CLS, GROUP, P₁..Pₙ]`, masked action head |
| 10 | **The env loop** | `SMModelEnv.step`: reward shaping, terminal bonus |
| 11 | **PPO** | rollouts vs iterations, GAE, the clipped update, entropy anneal |
| 12 | **train.py** | how it all wires together; reading a live run |

Run top-to-bottom. Cells that need a live Wolfram kernel are flagged **[needs kernel]**;
everything else runs against a `MockEngine` or the cached spectra with no kernel.


## 0 · Setup

### Running this notebook in the `uv` env

From the repo root:

```bash
uv sync --extra rl --extra notebook          # torch + jupyter + matplotlib into .venv
module load mathematica/15.0-fasrc01          # optional: a live Wolfram kernel (FASRC)
uv run jupyter lab notebooks/walkthrough.ipynb
```

`uv run jupyter …` launches Jupyter from the project's `.venv`, so `import sm_rl`
just works and the kernel picks up torch/scipy/numpy. If you started Jupyter some
other way, select the **`.venv`** Python as the notebook kernel.

**Live kernel vs offline.** A live Wolfram kernel is only needed to *compute new*
spectra. If `module load mathematica` has put `WolframKernel` on `PATH` (or
`$WOLFRAM_KERNEL` is set), the cells below use it. Otherwise they fall back to the
`MockEngine` (plumbing only, no real group theory) and to the **155k cached
spectra** from the training runs — enough to demonstrate every reward/analysis cell
without any kernel.


In [1]:
import os, sys, math, json, pickle
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # dodge OpenMP double-load abort
import numpy as np

# make the repo importable whether Jupyter's cwd is repo root or notebooks/
REPO = os.path.abspath(os.getcwd())
while REPO != "/" and not os.path.exists(os.path.join(REPO, "sm_rl")):
    REPO = os.path.dirname(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "tests"))         # for MockEngine
print("repo:", REPO)

from sm_rl.config import Config
cfg = Config()
print("n_u1 =", cfg.env.n_u1, "| d_max =", cfg.env.d_max, "| max_partons =", cfg.env.max_partons)
print("reward metric (dataclass default):", cfg.reward.metric,
      "| train.py default: f1")


repo: /Users/sambt/iaifi/ReLearning the SM
n_u1 = 4 | d_max = 3 | max_partons = 8
reward metric (dataclass default): count | train.py default: f1


### The `Config` object — every knob in one place

The framework is strictly layered, and **all** tunables live in dataclasses in
`sm_rl/config.py`. `Config` just aggregates four of them. This is what keeps the
physics / env / reward / algorithm layers decoupled: nothing reads a global, it
reads its slice of the config.


In [2]:
from dataclasses import fields
for name in ("engine", "groups", "env", "reward"):
    sub = getattr(cfg, name)
    print(f"\n### {name}  ({type(sub).__name__})")
    for f in fields(sub):
        v = getattr(sub, f.name)
        print(f"   {f.name:28s} = {v!r}")



### engine  (EngineConfig)
   kernel_path                  = '/Applications/Wolfram.app/Contents/MacOS/WolframKernel'
   mathematica_dir              = '/Users/sambt/iaifi/ReLearning the SM/Mathematica'
   groupmath_file               = 'GroupMath.m'
   spinsinglets_file            = 'spinSinglets.m'
   verbose                      = True

### groups  (GroupConfig)
   su_max_rank                  = 19
   so_max_rank                  = 10
   sp_max_rank                  = 10
   include_exceptionals         = True
   max_rep_dim                  = 248

### env  (EnvConfig)
   n_u1                         = 4
   max_partons                  = 8
   d_max                        = 3
   max_steps                    = 128
   charge_steps                 = (1.0, 1.0, 1.0, 0.5)
   charge_max                   = 6.0
   cpt_auto_conjugate           = True
   use_cursor                   = False
   reward_mode                  = 'delta'
   terminal_bonus               = 10.0
   min_partons_before_

## 1 · The Wolfram kernel — `physics/engine.py`

**Design invariant: the engine is the *only* thing that touches Wolfram.** Nothing
else in the codebase imports `wolframclient`. Everything downstream talks to the
abstract `Engine` interface, so the whole stack can run against `MockEngine` in
tests. `WolframEngine` is a thin, **fully memoized** wrapper: every method is a
pure function of its arguments, and `get_spins` (a GroupMath
`PermutationSymmetryOfInvariants` call) is the expensive one that caching exists
to avoid.

The `Engine` interface (abridged):

| method | returns | Wolfram call |
|--------|---------|--------------|
| `reps_up_to_dim(g, dmax)` | Dynkin labels of all irreps with dim ≤ dmax, **dimension-sorted** | `RepsUpToDimN` |
| `dim(g, rep)` | integer dimension | `DimR` |
| `dynkin_index(g, rep)` | Dynkin index | `DynkinIndex` |
| `casimir(g, rep)` | quadratic Casimir | `Casimir` |
| `conjugate_rep(g, rep)` | the dual rep (for CPT) | `ConjugateIrrep` |
| `conjugacy_class(g, rep)` | centre charge / N-ality vector | `ConjugacyClass` |
| `get_spins(g, reps, spins, flavs)` | `((2S+1, mult), …)` gauge singlets | `GetSpins` (spinSinglets.m) |

A `Rep` is a tuple of Dynkin labels: `(1,0)` is the SU(3) fundamental, `(0,1)` its
conjugate, `(1,1)` the adjoint (8).


In [ ]:
# Bootstrap an engine: prefer a live Wolfram kernel, fall back to MockEngine.
from sm_rl.physics.engine import WolframEngine
from mock_engine import MockEngine

def get_engine():
    # A kernel is usable if the engine can find it. cfg.engine.kernel_path is
    # already resolved by config.default_kernel_path() ($WOLFRAM_KERNEL -> PATH ->
    # macOS app bundle), and WolframEngine._preflight accepts it when it exists on
    # disk OR resolves on PATH. Mirror that exact test so the notebook goes live
    # whenever Mathematica is actually installed — not only when it's on $PATH.
    import shutil
    k = cfg.engine.kernel_path
    have_kernel = bool(os.environ.get("WOLFRAM_KERNEL")) or os.path.exists(k) or bool(shutil.which(k)) or any(
        shutil.which(n) for n in ("WolframKernel", "wolfram", "MathKernel", "math"))
    if have_kernel:
        try:
            eng = WolframEngine(cfg.engine).start()
            print("✓ live WolframEngine (GroupMath + spinSinglets loaded)")
            print("  kernel:", k)
            return eng, True
        except Exception as e:
            print("! WolframEngine failed to start:", e)
    print("… no kernel — using MockEngine (SU2/SU3 only, placeholder get_spins).")
    print("  Physics cells below are marked [needs kernel]; analysis cells use the cache.")
    return MockEngine().start(), False

eng, LIVE = get_engine()
print("LIVE =", LIVE)


In [ ]:
# Primitives on SU(3). With MockEngine these still work for SU2/SU3.
g = "SU3"
reps = eng.reps_up_to_dim(g, 248)          # dimension-ordered, trivial first
print(f"{g} reps up to dim 248 (Dynkin labels):")
for i, r in enumerate(reps[:6]):
    print(f"  index {i}: {r!s:8s} dim={eng.dim(g, r):3d}  "
          f"conj={eng.conjugate_rep(g, r)!s:8s}  N-ality={eng.conjugacy_class(g, r)}")
print("\nThe agent addresses reps by INDEX (0=trivial, 1=fundamental), never raw Dynkin.")


### 1.1 · What `GetSpins` actually computes — a read of `spinSinglets.m`

This is the physics heart. `spinSinglets.m` defines a single function, ported
verbatim from the `GT Invariants` notebook:

```mathematica
GetSpins[gaugeGroup_, partReps_, partSpins_, partFlavours_]
```

Given a multiset of matter fields — each with a **gauge rep**, a **spin** (`1` =
scalar/spin-0, `2` = fermion/spin-½, the GroupMath `2S+1` convention), and a
**flavour tag** — it returns the list of total spins at which those fields can
combine into a **gauge singlet** *consistent with spin-statistics*, together with
the multiplicity. Output rows are `{2S+1, multiplicity}`.

Here is the whole function, annotated:

```mathematica
maxSpin   = Count[partSpins, 2] + 1;            (* k fermions ⇒ max total 2S+1 = k+1  *)
testSpins = Table[i, {i, maxSpin, 1, -2}];      (* candidate 2S+1, stepping by 2       *)
numFlav   = Max[partFlavours];

(* Describe each field to GroupMath as an irrep of  gauge × SU2(spin) × U(1)^numFlav.
   The U(1) charges are a ONE-HOT of the field's flavour — this is the trick that
   makes otherwise-identical fields distinguishable ("dummy quantum number"), so the
   invariant search knows which fields are truly the same. *)
product  = Table[Join[{partReps[[i]], partSpins[[i]]},
                      IdentityMatrix[numFlav][[partFlavours[[i]]]]], {i, Length[partReps]}];
refCharge = -Total[product[[All, 3 ;; All]]];   (* charges of the compensating "sink" field *)
fullGroup = Join[{gaugeGroup, SU2}, Table[U1, {i, numFlav}]];

Do[                                              (* for each candidate total spin tspin *)
   tProd = Append[product, Join[{1, tspin}, refCharge]];   (* add a gauge-singlet(=1) probe *)
   tPerm = PermutationSymmetryOfInvariants[fullGroup, tProd];
   If[Length[tPerm[[2]]] > 0, AppendTo[possibleSpins, {tspin, tPerm[[2]]}]];  (* invariant exists? *)
   ...
, {tspin, testSpins}];

(* Keep only invariants whose PERMUTATION SYMMETRY under exchange of identical
   fields matches their statistics:
     fermions  → fully ANTI-symmetric  (Young diagram = one column)
     bosons    → fully  symmetric      (Young diagram = one row) *)
Do[ checks = True;
    Do[ op = tspin[[2,1,1,l]];
        If[ partSpins[[labels[[l,1]]]] == 2,
            If[Length[labels[[l]]] != Length[op],   checks = False],   (* need a full column *)
            If[Length[labels[[l]]] != op[[1]],      checks = False] ], (* need a full row    *)
      {l, Length[labels]}];
    If[checks, AppendTo[allowedSpins, {tspin[[1]], tspin[[-1,-1,-1]]}]];  (* {2S+1, mult} *)
, {tspin, possibleSpins}];
allowedSpins
```

**The four ideas that matter:**

1. **Singlet = invariant of the full group.** Bound states are colour singlets; the
   probe field with gauge rep `1` (the singlet) and compensating `refCharge` lets
   `PermutationSymmetryOfInvariants` decide whether the partons can close into an
   overall invariant of `gauge × SU2 × U(1)^flavour`.
2. **Spin rides along as an SU(2) factor.** By tensoring in an explicit `SU2` and
   scanning candidate `2S+1`, the *same* invariant machinery that enforces gauge
   singlet-ness also enforces the spin structure. That's why a hadron's spin comes
   out of a group-theory calculation.
3. **Flavour one-hots = which fields are identical.** Distinct partons get distinct
   U(1) charges so the permutation analysis only symmetrises over *genuinely*
   identical fields.
4. **Spin-statistics is the filter.** Not every gauge+spin invariant is physical —
   only those with the right exchange symmetry. Fermions must land in a fully
   antisymmetric (single-column) representation of the identical-field permutation
   group; bosons in a fully symmetric (single-row) one. `GetSpins` keeps exactly
   those.

The Python `get_spins` is a one-line shim that formats the arguments, calls this,
and memoizes the result.


In [ ]:
# [needs kernel] Run GetSpins directly. Three quarks (fundamental, spin-1/2), all
# distinct flavours -> this is the BARYON channel (qqq). We expect spin-1/2 and 3/2
# (the octet/decuplet story), i.e. 2S+1 in {2, 4}.
if LIVE:
    fund = eng.reps_up_to_dim("SU3", 248)[1]          # (1,0)
    out = eng.get_spins("SU3", [fund, fund, fund], [2, 2, 2], [1, 2, 3])
    print("qqq singlets (2S+1, multiplicity):", out)
    print("  -> spins:", [f"{(l-1)/2:g}" for l, _ in out])
    # And a meson channel: quark + antiquark (conjugate rep).
    anti = eng.conjugate_rep("SU3", fund)             # (0,1)
    print("q-qbar singlets:", eng.get_spins("SU3", [fund, anti], [2, 2], [1, 2]))
else:
    print("[needs kernel] skipped — load mathematica to run GetSpins on SU(3).")
    print("MockEngine.get_spins only returns a singlet for a lone trivial rep (plumbing test).")


## 2 · The gauge-group ladder — `physics/groups.py`

The agent doesn't pick a group from a flat list; it **walks a graph**. `GroupLadder`
enumerates candidate groups as GroupMath strings and defines two move families:

- **RANK step** — next/previous `N` within a family: `SU3 → SU4`, `SO7 → SO9`, …
- **TYPE step** — switch classical family `SU → SO → SP → EX`, landing on the group
  of *nearest Lie rank* in the new family.

Two design invariants live here:

- **Reps are dimension-ordered indices**, not raw Dynkin labels. Index 0 = trivial,
  1 = fundamental. A parton stores an *index*.
- **Group changes clamp the index down** to the new group's admissible reps
  (dim ≤ `max_rep_dim`, default 248). So "rep 3 of SU(3)" becomes "rep 1 of E8" if
  E8 only has the trivial rep and the 248 under the cap.


In [ ]:
from sm_rl.physics.groups import GroupLadder, FAMILY_ORDER
ladder = GroupLadder(eng, cfg.groups)
print("families in scope:", ladder.family_order, "  (order:", FAMILY_ORDER, ")")
print("total candidate groups:", len(ladder.specs))
print("first few per family:")
for fam in ladder.family_order:
    names = [s.gm for s in ladder.families[fam][:5]]
    print(f"   {fam}: {names} ...")
print("\ndefault episode start group:", ladder.default.gm, "(smallest simple group)")


In [ ]:
# Transitions from SU3. type_step cycles families; rank_step returns None at a boundary.
su3 = ladder.spec("SU3")
print("SU3:  N =", su3.N, " lie_rank =", su3.lie_rank, " family =", su3.family)
print("  RANK +1 ->", ladder.rank_step(su3, +1).gm)
print("  RANK -1 ->", ladder.rank_step(su3, -1).gm, "   (SU3 is one RANK_UP from SU2!)")
for d in (+1, -1):
    print(f"  TYPE {d:+d} -> {ladder.type_step(su3, d).gm}  (nearest Lie rank in next family)")


> **Note for the training runs.** SU(3) — the answer — is *trivially reachable*: a
> single `RANK_UP` from the SU(2) start. So the challenge was never reachability;
> it was that the old reward made SU(3) score *worst* of all groups (see §7).


In [ ]:
# Rep-index clamping on a group change. This is the "reps round down" behaviour.
if LIVE:
    for gname in ("SU3", "E8", "G2"):
        spec = ladder.spec(gname)
        n = ladder.n_reps(spec)
        print(f"{gname:4s}: {n:3d} reps up to dim {cfg.groups.max_rep_dim}. "
              f"clamp(index=5) -> {ladder.clamp_index(spec, 5)}  "
              f"(rep {ladder.rep_at(spec, 5)})")
else:
    spec = ladder.spec("SU3")
    print("SU3 reps (mock):", ladder.n_reps(spec),
          "| clamp(index=5) ->", ladder.clamp_index(spec, 5))
    print("[needs kernel] E8/G2 rep counts need a live kernel.")


## 3 · State & actions — `env/state.py`, `env/actions.py`

**State** is a `Model`: a gauge-group string plus an *ordered* list of `Parton`s.

- The order is **bookkeeping, not physics** — it exists so the policy (with
  positional encodings) can point at "the most-recently-added" parton as the default
  edit target.
- Each parton carries a **unique `flavour` tag**. Look-alike partons are never
  merged; the flavour is the "dummy quantum number" that keeps `u` and a second
  identical `u'` distinct (and, as we saw, becomes a U(1) one-hot inside `GetSpins`).
- A parton stores `rep_index` (into the group's dim-ordered reps), `spin` (1 or 2),
  and a length-`n_u1` `charges` list.


In [ ]:
from sm_rl.env.state import Model, Parton
m = Model(group="SU3")
m.partons.append(Parton(rep_index=1, spin=2, charges=[2.,1.,0.,0.5], flavour=m.new_flavour()))
m.partons.append(Parton(rep_index=1, spin=2, charges=[-1.,1.,0.,-0.5], flavour=m.new_flavour()))
print("model:", m.describe())
print("n =", m.n, "| flavours:", [p.flavour for p in m.partons], "(always distinct)")
print("edit target (last parton):", m.edit_target(use_cursor=False).charges)
print("round-trips through JSON:", Model.from_dict(m.to_dict()).describe() == m.describe())


### The action space

`ActionSpace` is a **flat, maskable** list of small deterministic edits. Every
transition is **pure Python — no engine call** (the expensive spectrum evaluation
happens later, in the reward). The actions:

`STOP`, `TYPE_±`, `RANK_±`, `ADD`, `REMOVE_LAST`, `SPIN_TOGGLE`, `REP_±`, and
`CHARGE{c}_±` for each U(1) column. (An optional movable cursor adds
`CURSOR_PREV/NEXT` when `use_cursor` is set.)

A **mask** marks which actions are valid in the current state — e.g. you can't
`REMOVE_LAST` from an empty model, can't step a charge past `charge_max`, can't
`RANK_UP` past a family boundary. The policy sets masked-out logits to `-∞`, so
invalid actions are never sampled.


In [ ]:
from sm_rl.env.actions import ActionSpace
aspace = ActionSpace(cfg.env, ladder)
print(f"{len(aspace)} actions:")
for i, a in enumerate(aspace.actions):
    print(f"  {i:2d} {a.name:14s} kind={a.kind:8s} param={a.param}")


In [ ]:
# Apply/mask demo. Note apply() returns a NEW model (input not mutated).
m0 = Model("SU3")
idx = aspace.index
mask0 = aspace.mask(m0)
avail = [a.name for a, ok in zip(aspace.actions, mask0) if ok]
print("empty SU3 model — valid actions:", avail)
print("  STOP valid on empty model?", bool(mask0[idx["STOP"]]),
      f"(min_partons_before_stop = {cfg.env.min_partons_before_stop})")

m1 = aspace.apply(m0, idx["ADD"])              # add a default parton (fund, scalar, q=0)
m1 = aspace.apply(m1, idx["SPIN_TOGGLE"])      # -> fermion
m1 = aspace.apply(m1, idx["CHARGE0_UP"])       # bump EM by charge_steps[0]
print("\nafter ADD, SPIN_TOGGLE, CHARGE0_UP:", m1.describe())


## 4 · From a model to its spectrum — `physics/spectrum.py`

Three steps turn a `Model` into an `(M, n_u1+1)` array of hadron rows
`[charges…, 2S+1]`:

**(a) `resolve_fields` — CPT expansion.** Each parton's `rep_index` is resolved to
Dynkin labels, and (if `cpt_auto_conjugate`) its **conjugate** is appended: dual
rep, negated charges. This is how `(u,d,s)` alone yields mesons *and* antibaryons —
"antiparticles implied by CPT". A genuinely self-conjugate neutral field isn't
double-counted.

**(b) `compute_spectrum` — operator enumeration.** Enumerate every multiset of up
to `d_max` resolved fields (`combinations_with_replacement`), call `get_spins` on
each, and for every returned `(2S+1, mult)` emit `mult` rows carrying that
operator's summed charges and spin label. This is the loop that spends Mathematica
time — `O(#fields^d_max)` `get_spins` calls.

**(c) `canonical_key` — an order-invariant physics fingerprint.** Sorted resolved
fields + group + rounding. Two models that differ only in parton order (or that
CPT-resolve to the same field set) share a key — and therefore a cache entry.


In [ ]:
from sm_rl.physics.spectrum import resolve_fields, canonical_key, spectrum_of, normalize_spectrum

# One 'u'-like quark in SU3; watch CPT append its conjugate.
mu = Model("SU3")
mu.partons.append(Parton(1, 2, [2.,1.,0.,0.5], mu.new_flavour()))
fields = resolve_fields(mu, ladder, cpt_auto_conjugate=True)
print("resolved fields (rep, spin, charges) — CPT doubled 1 parton -> 2 fields:")
for f in fields:
    print("  ", f)
print("\ncanonical key (order-invariant):")
print("  ", canonical_key(mu, ladder, True))


In [ ]:
# [needs kernel] Full spectrum of the single quark (mesons q-qbar only, no baryon
# since d_max=3 but you need 3 quarks for an SU(3) baryon and we have just one q).
if LIVE:
    spec = spectrum_of(eng, ladder, mu, cfg.env, cache={})
    print("spectrum shape:", spec.shape, " columns = [EM, B, S, I3, 2S+1]")
    print(spec)
else:
    print("[needs kernel] spectrum_of needs a live kernel. We'll load CACHED spectra next.")


## 5 · The spectrum cache — `physics/cache.py`

Every cache **miss costs a Mathematica evaluation**, so persisting spectra across
runs (and sharing them across data-parallel workers) is a large saving.
`SpectrumCache` is a `dict` subclass keyed on `canonical_key`:

- **Persistent** — pickles to disk; reloads on construction.
- **Auto-flush** every `flush_every` new entries.
- **Merge-on-flush** — reads the on-disk file, merges, writes to a temp file, then
  `os.replace` (atomic). Multiple workers pointed at one path accumulate each
  other's results safely. Because spectra are deterministic, the rare key collision
  is harmless (last-writer-wins).

The training runs share one file on lab storage, seeded from the first `cluster`
run. **Crucially, the key is pure physics** — independent of reward/normalization —
so the cache stays valid across every reward change we make in §7.


In [ ]:
from sm_rl.physics.cache import SpectrumCache

SHARED = "/n/holystore01/LABS/iaifi_lab/Users/sambt/sm_rl_cache/spectra.pkl"
CACHE_PATH = SHARED if os.path.exists(SHARED) else os.path.join(REPO, "runs/cluster/cache.pkl")
have_cache = os.path.exists(CACHE_PATH)
print("cache path:", CACHE_PATH, "| exists:", have_cache)

if have_cache:
    cache = SpectrumCache(CACHE_PATH)          # loads from disk
    print(f"loaded {len(cache):,} cached spectra")
    k = next(iter(cache))
    print("\nexample key:", k[0], "with", len(k[1]), "resolved fields")
    print("its spectrum shape:", cache[k].shape)
else:
    cache = SpectrumCache(None)
    print("no cache on disk — will compute live if a kernel is present.")


> The cache is a plain `dict` under the hood, so anywhere the code wants a cache it
> can also take `{}` (in-memory) — handy for tests. `cache.stats()` reports the
> hit-rate you see in the training logs (`hit_rate=0.99…`).


## 6 · The target — `reward/target.py`

The thing to rediscover: **SU(3) with `(u, d, s)` quarks**. `build_target`
constructs that model and runs it through the *same* `spectrum_of`, so the target is
guaranteed self-consistent with predictions (the true model is literally the argmax
of the reward).

**Charge convention** `[EM, B, S, I3]`, scaled-integer:

| charge | stored as | note |
|--------|-----------|------|
| EM  | physical × 3 | so up-quark EM = +2 (physical +2/3) |
| B   | physical × 3 | baryon number |
| S   | physical × 3 | strangeness |
| I₃  | in halves | already physical |

Antiquarks are **not** listed — CPT supplies them. So the content is just `(u,d,s)`.


In [ ]:
from sm_rl.reward.target import QUARK_CHARGES
print("quark charges [EM, B, S, I3] (EM/B/S are physical x3):")
for q in ("u", "d", "s"):
    print(f"   {q}: {QUARK_CHARGES[q]}")

# The target spectrum: build live, else recover it from the cache by its physics key.
def load_target():
    if LIVE:
        from sm_rl.reward.target import build_target
        t, tm = build_target(eng, ladder, cfg.env, cache=cache if have_cache else {})
        return t
    # offline: the (u,d,s) SU3 model has a known canonical key -> pull from cache
    from sm_rl.reward.target import build_target  # only to get FERMION/fund via a mock? no—reconstruct
    tm = Model("SU3")
    for q in ("u","d","s"):
        tm.partons.append(Parton(1, 2, list(QUARK_CHARGES[q]), tm.new_flavour()))
    key = canonical_key(tm, ladder, cfg.env.cpt_auto_conjugate)
    if have_cache and key in cache:
        return cache[key]
    raise RuntimeError("target not in cache and no live kernel")

target = load_target()
print("\ntarget spectrum:", target.shape, "hadron rows (the 54-state light-hadron target)")
print("charge |max| per column [EM,B,S,I3]:", np.abs(target[:, :4]).max(axis=0))
print("distinct 2S+1 spin labels:", sorted(set(target[:, 4].tolist())))


## 7 · Reward metrics — `reward/*.py`

A `RewardMetric` scores a predicted spectrum against the target; **higher is
better**. The interface (`reward/base.py`) is pluggable via `make_metric`, and each
metric owns its own charge **normalization** (applied before matching).

Three metrics ship:

1. **`count`** — exact quantum-number signatures; `score = -(missing + extra)`.
   No partial credit for near-misses.
2. **`hungarian`** — optimal one-to-one partial matching with *graded* costs.
   Looks principled. **It is broken as a learning signal** — shown below.
3. **`f1`** — quality-weighted F1, bounded `[0,1]`, `1.0` iff exact. **The fix**
   and the current `train.py` default.

Normalization matters more than it looks: charges are stored ×3, so the target's
charge quantum is **3**.


In [ ]:
from sm_rl.reward import make_metric

def scored(metric_name, normalization, **extra):
    c = Config().reward
    c.metric = metric_name
    c.normalization = normalization
    for k, v in extra.items(): setattr(c, k, v)
    return make_metric(c, cfg.env.n_u1)

m_count = scored("count", "gcd")
m_hung  = scored("hungarian", "none")
m_f1    = scored("f1", "physical")

# sanity: every metric scores the target against itself at its "perfect" value
for name, mt in [("count", m_count), ("hungarian", m_hung), ("f1", m_f1)]:
    r = mt.match(target.copy(), target)
    print(f"{name:10s} target-vs-target: score={r.score:+.4f}  exact={r.exact}")


### 7.1 · Why `hungarian + none` is broken

The Hungarian score is `1 − total_cost / (lam_miss · |target|)`. Its per-unmatched
prediction penalty (`lam_spur`) is **additive and unbounded**, while the number of
target states it can match **saturates**. So the score becomes a *decreasing linear
function of how many hadrons a model predicts*. The reward-maximising model predicts
**nothing** — and SU(3), the only group that forms many singlets at `d_max=3`, is
penalised hardest.

Let's measure it on the cached SU(3) models. (Needs the cache; no kernel.)


In [ ]:
if not have_cache:
    print("[needs cache] skip — no cached spectra available.")
else:
    su3 = [(k, v) for k, v in cache.items()
           if k[0] == "SU3" and isinstance(v, np.ndarray) and v.ndim == 2 and v.shape[1] == 5]
    rng = np.random.default_rng(0)
    sample = [su3[i] for i in rng.choice(len(su3), min(3000, len(su3)), replace=False)]
    npred = np.array([len(v) for _, v in sample])
    s_h = np.array([m_hung.match(v.copy(), target).score for _, v in sample])
    s_f = np.array([m_f1.match(v.copy(),   target).score for _, v in sample])
    empty_h = m_hung.match(np.zeros((0,5)), target).score
    empty_f = m_f1.match(np.zeros((0,5)),   target).score
    print(f"cached SU(3) models sampled: {len(sample)}")
    print(f"  hungarian/none:  corr(n_pred, score) = {np.corrcoef(npred, s_h)[0,1]:+.3f}"
          f"   empty-model score = {empty_h:+.3f}   frac beating empty = {np.mean(s_h>empty_h):.1%}")
    print(f"  f1/physical:     corr(n_pred, score) = {np.corrcoef(npred, s_f)[0,1]:+.3f}"
          f"   empty-model score = {empty_f:+.3f}   frac beating empty = {np.mean(s_f>empty_f):.1%}")
    print("\n-> hungarian: predicting more hadrons LOWERS the score (corr ~ -1);"
          " the empty model beats most real SU(3) models.")


In [ ]:
# The mechanism, tabulated: matched saturates, extra grows 1:1 and is charged linearly.
if have_cache:
    print("hungarian/none  — SU(3) score vs number of predicted hadrons:")
    print(f"{'n_pred bin':>12} {'n':>5} {'score':>8} {'matched':>8} {'extra':>7}")
    rows = np.array([(len(v), m_hung.match(v.copy(), target).score,
                      m_hung.match(v.copy(), target).info['matched'],
                      m_hung.match(v.copy(), target).info['extra']) for _, v in sample])
    for lo, hi in [(0,10),(10,20),(20,40),(40,60),(60,120),(120,10000)]:
        sel = rows[(rows[:,0]>=lo)&(rows[:,0]<hi)]
        if len(sel):
            print(f"{f'{lo}-{hi}':>12} {len(sel):>5} {sel[:,1].mean():>+8.3f} "
                  f"{sel[:,2].mean():>8.1f} {sel[:,3].mean():>7.1f}")


### 7.2 · The fix — `F1Reward` (`reward/f1.py`)

Two independent changes, each fixing one half of the pathology:

**Bounded objective.** Score a quality-weighted F1 instead of an additive cost:

$$\text{gain}(i,j)=\max\!\Big(0,\ 1-\frac{\text{cost}(i,j)}{\text{match\_radius}}\Big)\in[0,1],\qquad
\text{score}=\frac{2\cdot\text{soft\_matched}}{n_\text{pred}+n_\text{target}}\in[0,1].$$

Overproduction now dilutes *precision* sub-linearly instead of adding unbounded
cost. Score is bounded, `1.0` iff exact, `0.0` for both the empty prediction and for
unbounded spam. Matching **more** target states always helps — so adding the quark
that opens the baryon sector is rewarded.

**`physical` normalization.** Divide charges by `(3,3,3,1)` (the `charge_scale`) to
undo the ×3 convention, so the charge quantum is **1**. This is deterministic
(unlike `gcd`) and — critically — puts one-unit near-misses *inside* the match
radius. Under `none`, the quantum is 3 ≥ radius, so a one-unit charge error earns
**zero** partial credit: a graded metric with no gradient.


In [ ]:
# Does adding a quark help now? Group cached SU(3) models by parton count.
if have_cache:
    byn = {}
    for k, v in cache.items():
        if k[0]=="SU3" and isinstance(v, np.ndarray) and v.ndim==2 and v.shape[1]==5:
            byn.setdefault(len(k[1])//2, []).append(v)   # len(fields)//2 = #partons (CPT doubles)
    print("mean score by number of partons (SU(3)):")
    print(f"{'n_partons':>10} {'hungarian/none':>16} {'f1/physical':>14}")
    for n in sorted(byn)[:6]:
        vs = byn[n][:250]
        h = np.mean([m_hung.match(v.copy(), target).score for v in vs])
        f = np.mean([m_f1.match(v.copy(),   target).score for v in vs])
        print(f"{n:>10} {h:>+16.3f} {f:>+14.3f}")
    print("\n-> hungarian: every added quark HURTS.  f1: adding quarks climbs toward the answer.")


In [ ]:
# The headline table: is SU(3) rewarded, over the whole group ladder?
if have_cache:
    from collections import defaultdict
    rng = np.random.default_rng(1)
    keys = list(cache.keys()); pick = [keys[i] for i in rng.choice(len(keys), min(12000, len(keys)), replace=False)]
    gh, gf = defaultdict(list), defaultdict(list)
    for k in pick:
        v = cache[k]
        if isinstance(v, np.ndarray) and v.ndim==2 and v.shape[1]==5:
            gh[k[0]].append(m_hung.match(v.copy(), target).score)
            gf[k[0]].append(m_f1.match(v.copy(),   target).score)
    groups = [g for g in gf if len(gf[g])>=50]
    rank_h = sorted(groups, key=lambda g: -np.mean(gh[g]))
    rank_f = sorted(groups, key=lambda g: -np.mean(gf[g]))
    print(f"groups scored: {len(groups)}")
    print(f"  SU(3) rank under hungarian/none: {rank_h.index('SU3')+1}/{len(groups)}"
          f"   (mean {np.mean(gh['SU3']):+.3f})")
    print(f"  SU(3) rank under f1/physical:    {rank_f.index('SU3')+1}/{len(groups)}"
          f"   (mean {np.mean(gf['SU3']):+.3f})")
    print("\n-> the reward fix moves SU(3) from near-worst to top-tier. That is the whole game.")


### 7.3 · The regression tests that pin this

`tests/test_core.py` now asserts the reward contract so this class of bug can't come
back: the target is the unique global max; a partially-correct model beats the empty
model; overproduction stays bounded in `[0,1]`; and near-misses earn graded credit
only under `physical` normalization. Run them any time with:

```bash
uv run python tests/test_core.py
```


## 8 · Tokenizer — `models/tokenizer.py`

The policy can't eat a `Model` directly; the tokenizer turns an observation into
fixed-width tensors. The key design choice: **reps are described by GROUP-INVARIANT
physical scalars**, never raw Dynkin labels —

$$\big[\log(1+\dim),\ \text{Dynkin index},\ \text{Casimir},\ \text{N-ality}_0,\ \text{N-ality}_1\big]$$

— so a rep token means the same thing across groups and survives a group change.
Each parton token = these rep scalars + a spin one-hot + its charges + an
"is-edit-target" flag. The gauge group gets its own token (family one-hot + rank +
`log(#reps)`).


In [ ]:
from sm_rl.models.tokenizer import Tokenizer, GROUP_FEAT, REP_FEAT
tok = Tokenizer(eng, ladder, cfg.env.n_u1)
print("feature widths:  group =", GROUP_FEAT, " rep scalars =", REP_FEAT,
      " parton =", tok.parton_feat, "(rep + spin2 + charges + edit)")

# Build a 2-parton SU3 model and featurize it.
mm = Model("SU3")
mm.partons.append(Parton(1, 2, [2.,1.,0.,0.5],  mm.new_flavour()))
mm.partons.append(Parton(1, 2, [-1.,1.,0.,-0.5], mm.new_flavour()))
mask = aspace.mask(mm)
feat = tok.featurize({"model": mm, "mask": mask}, use_cursor=False)
print("\ngroup_feat :", np.round(feat['group_feat'], 3))
print("parton_feat:", feat['parton_feat'].shape, "(n_partons x parton_feat)")
print("  parton 0 :", np.round(feat['parton_feat'][0], 3))
print("  (last col = 1.0 => this is the edit target)")


In [ ]:
# collate() pads a list of models into a batch the transformer consumes.
batch = tok.collate([feat, tok.featurize({"model": Model("SU2"), "mask": aspace.mask(Model("SU2"))}, False)])
for k, v in batch.items():
    print(f"  {k:14s} {getattr(v,'shape',None)}  dtype={getattr(v,'dtype',None)}")
print("pad_mask True == padding (shorter model padded up to the batch max).")


## 9 · The transformer actor-critic — `models/transformer.py`

A small Transformer encoder over the token sequence **`[CLS, GROUP, P₁…Pₙ]`**:

- `CLS` — a learned pooling token; its output drives both heads.
- `GROUP` — the group features, projected to `d_model`.
- `Pᵢ` — parton tokens: features projected + **sinusoidal positional encoding**
  (so the net can track ordering / "last added") + a learned segment embedding.

Two heads read the `CLS` output: a **masked action head** (invalid logits → `-∞`, so
sampling respects the mask) and a **scalar value head** (the critic). The network is
deliberately small — few labelled rewards. It is **algorithm-agnostic**: it emits
`logits, value` and knows nothing about PPO.


In [ ]:
import torch
from sm_rl.models.transformer import ActorCritic, to_torch

policy = ActorCritic(GROUP_FEAT, tok.parton_feat, len(aspace),
                     max_partons=cfg.env.max_partons + 4)
nparam = sum(p.numel() for p in policy.parameters())
print(f"ActorCritic: {nparam:,} params  (d_model=128, 3 layers, 4 heads)")

tb = to_torch(batch, "cpu")
with torch.no_grad():
    logits, value = policy(tb)
print("\nlogits:", tuple(logits.shape), " value:", tuple(value.shape))
print("masked logits for model 0 (note -1e9 on invalid actions):")
for name, lg, ok in zip([a.name for a in aspace.actions], logits[0].tolist(), batch['action_mask'][0]):
    flag = "" if ok else "  <- masked"
    print(f"   {name:14s} {lg:+9.3f}{flag}")


In [ ]:
# act() samples a valid action; evaluate() gives log-prob/entropy/value for an update.
with torch.no_grad():
    a, logp, v = policy.act(tb)
print("sampled actions:", a.tolist(), " (always valid — masked dist)")
print("log-probs:", np.round(logp.tolist(), 3), " values:", np.round(v.tolist(), 3))
lp, ent, val = policy.evaluate(tb, a)
print("entropy per state:", np.round(ent.tolist(), 3), "(higher = more exploratory)")


## 10 · The environment loop — `env/sm_env.py`

`SMModelEnv` ties it together into a gym-like `reset/step`. Per step:

1. `ActionSpace.apply` — edit the model (pure Python).
2. `spectrum_of` — compute (or cache-hit) the new spectrum.
3. `RewardMetric.match` — score it against the target.
4. Emit reward.

**Reward shaping** (`reward_mode`):

- `absolute`: `rₜ = score(sₜ)` — dense, but can reward stalling in a good state.
- `delta` (default): `rₜ = score(sₜ) − score(sₜ₋₁)` — potential-based; the returns
  **telescope** to (final score − initial score), so there's no incentive to loiter.

On `STOP`, if the model exactly matches the target, a `terminal_bonus` is added.


In [ ]:
from sm_rl.env import SMModelEnv

# Build the env. Live -> real target/spectra; offline -> pass the cached target in.
if LIVE:
    env = SMModelEnv(eng, cfg, cache=cache if have_cache else {})
else:
    env = SMModelEnv(eng, cfg, target=target, cache=cache if have_cache else {})
print("env ready. target rows:", len(env.target), "| num_actions:", env.num_actions)
print("reward_mode:", cfg.env.reward_mode, "| terminal_bonus:", cfg.env.terminal_bonus)


In [ ]:
# Drive a few steps by hand and watch score, delta-reward, and the running spectrum size.
obs, info = env.reset(start_group="SU3")
print(f"reset: score={info['score']:+.4f}\n")
rng = np.random.default_rng(2)
for t in range(8):
    mask = env.action_mask()
    a = int(rng.choice(np.flatnonzero(mask)))     # random *valid* action
    obs, r, term, trunc, info = env.step(a)
    print(f"step {t}: {info['action']:14s} r={r:+.4f} score={info['score']:+.4f} "
          f"n_pred={info.get('n_pred','-')} {'STOP' if term else ''}")
    if term or trunc:
        break


## 11 · PPO — `algos/ppo.py`, `algos/base.py`

### Rollouts vs iterations (the distinction you asked about)

- A **rollout** is `rollout_len` *environment steps* collected with the current
  frozen policy (`collect_rollout` in `algos/base.py`). Episodes auto-reset inside a
  rollout, so one rollout usually spans many short episodes.
- An **iteration** is *one* rollout **plus** one policy update on that data. The
  update does `epochs` passes over the rollout in minibatches of `minibatch_size`.

So `--iters N --rollout L` collects `N·L` total environment steps. **Iterations are
not comparable across different `rollout_len`** — that's why the earlier run's 40k
iters at rollout 64 (= 2.56M steps) equals only 2,500 iters at rollout 1024.

### GAE

`compute_gae` turns per-step rewards + value estimates into advantages (how much
better an action was than the critic expected) and returns (value targets), with the
usual `gamma`/`lambda` bias-variance knobs.

### The clipped update

For each minibatch: recompute log-probs under the current policy, form the
probability ratio vs the collection-time policy, and optimise the **clipped
surrogate** so no single step moves the policy too far, plus a value-regression loss
and an **entropy bonus** (keeps exploration alive). Entropy coefficient is
**linearly annealed** `ent_coef → ent_coef_final` over `ent_anneal_iters`, keyed on
`iters_done` so it's resume-safe.


In [ ]:
from sm_rl.algos import PPO, PPOConfig
from sm_rl.algos.base import collect_rollout, compute_gae

pcfg = PPOConfig(rollout_len=64, minibatch_size=32, start_group="SU3")   # tiny, for demo
print("PPOConfig:", {k: getattr(pcfg, k) for k in
      ('lr','gamma','gae_lambda','clip','epochs','minibatch_size','rollout_len',
       'ent_coef','ent_coef_final','ent_anneal_iters')})

roll = collect_rollout(env, policy, tok, cfg.env.use_cursor, pcfg.rollout_len,
                       start_group="SU3", device="cpu")
print(f"\nrollout: {len(roll['actions'])} steps spanning {len(roll['ep_returns'])} episodes")
print("  -> ~%.1f steps/episode" % (pcfg.rollout_len / max(1, len(roll['ep_returns']))))
print("  best score seen in rollout:", round(roll['best_score'], 4))


In [ ]:
# GAE on the rollout.
adv, ret = compute_gae(roll['rewards'], roll['values'], roll['dones'],
                       roll['last_value'], pcfg.gamma, pcfg.gae_lambda)
print("advantages:", np.round(adv[:6], 3), "...")
print("returns   :", np.round(ret[:6], 3), "...")

# Entropy anneal schedule (resume-safe: a pure function of iters_done).
ppo = PPO(env, policy, tok, pcfg)
print("\nentropy coef vs iters_done (anneal horizon =", pcfg.ent_anneal_iters, "):")
for it in (0, 1000, 2500, 5000, 6000):
    ppo.iters_done = it
    print(f"   iter {it:>5}: ent_coef = {ppo._current_ent_coef():.4f}")
ppo.iters_done = 0


In [ ]:
# One full training iteration (rollout + update). Watch the stats dict.
stats = ppo._update(roll)
print("update stats:", {k: round(v, 4) for k, v in stats.items()})
print("\nThese map to the training log line:")
print("  iter N mean_r=… ep_ret=… best=… ent=<entropy> ec=<ent_coef> "
      "pg=<pg_loss> vf=<v_loss> eps=<n_episodes> exact=… hit_rate=<cache>")


> **What the earlier failed runs looked like here.** With the broken reward, the
> advantage signal was noise, entropy collapsed to exactly `0.0000`, and every
> episode became the same 4-step "predict almost nothing" model. `best` was set by
> the *randomly initialised* policy at iteration 0 and never improved. The f1 fix is
> what gives these quantities something real to climb.


## 12 · Putting it together — `scripts/train.py`

The full runner wires all of the above and adds file logging + checkpointing +
the persistent cache. The essential flow:

```python
cfg = Config()
cfg.reward.metric = "f1"; cfg.reward.normalization = "physical"   # the defaults now
eng  = WolframEngine(cfg.engine).start()
cache = SpectrumCache(cache_path, flush_every=200)                # $SM_RL_CACHE, shared
env  = SMModelEnv(eng, cfg, cache=cache)
tok  = Tokenizer(eng, env.ladder, cfg.env.n_u1)
policy = ActorCritic(GROUP_FEAT, tok.parton_feat, env.num_actions, ...)
ppo  = PPO(env, policy, tok, PPOConfig(rollout_len=..., start_group="SU3"))
ppo.train(iters, logger=logger, ckpt_every=...)                   # rollout+update loop
```

**Cache path precedence:** `--cache` > `$SM_RL_CACHE` > `<run_dir>/cache.pkl`. The
SLURM scripts point `$SM_RL_CACHE` at the shared lab-storage file so every run pools
into it and nothing is ever recomputed.

Typical invocation (what the running jobs use):

```bash
uv run python scripts/train.py --iters 5000 --ckpt-every 50 \
    --metric f1 --normalization physical --run-name f1fix --start-group SU3 --seed 0
```

Each run writes `runs/<name>_<ts>/` with `log.txt`, `metrics.csv`, `config.json`,
`best_model.json`, `checkpoints/`, and (unless shared) `cache.pkl`. Resume with
`--run-dir <dir> --resume latest`.


In [ ]:
# Read a live/finished run's metrics, if any exist. (Pure pandas-free CSV read.)
import glob, csv
runs = sorted(glob.glob(os.path.join(REPO, "runs", "f1fix*")) +
              glob.glob(os.path.join(REPO, "runs", "tier1*")))
print("runs found:")
for d in runs:
    log = os.path.join(d, "log.txt")
    best = os.path.join(d, "best_model.json")
    n = sum(1 for _ in open(log)) if os.path.exists(log) else 0
    b = json.load(open(best))["describe"] if os.path.exists(best) else "-"
    print(f"  {os.path.basename(d):32s} log_lines={n:>5}  best={b}")


In [ ]:
# Plot the learning curve of the most recent run (best_score & entropy vs iter).
import matplotlib
matplotlib.use("Agg")   # safe in headless; comment out for inline
import matplotlib.pyplot as plt

cand = [d for d in runs if os.path.exists(os.path.join(d, "metrics.csv"))]
if cand:
    d = cand[-1]
    rows = list(csv.DictReader(open(os.path.join(d, "metrics.csv"))))
    it   = [int(r["iter"]) for r in rows]
    best = [float(r["best_score"]) for r in rows]
    ent  = [float(r["entropy"]) for r in rows]
    epr  = [float(r["ep_return"]) for r in rows]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
    ax[0].plot(it, best); ax[0].set_title("best_score"); ax[0].set_xlabel("iter")
    ax[1].plot(it, epr);  ax[1].set_title("ep_return");  ax[1].set_xlabel("iter")
    ax[2].plot(it, ent);  ax[2].set_title("entropy");    ax[2].set_xlabel("iter")
    fig.suptitle(os.path.basename(d)); fig.tight_layout()
    out = os.path.join(REPO, "notebooks", "last_run_curves.png")
    fig.savefig(out, dpi=110); print("saved", out)
    plt.show()
    print(f"\n{os.path.basename(d)}: final best={best[-1]:+.3f} at iter {it[-1]}")
    print("Healthy signs to look for: best climbing (not flat from iter 0), "
          "entropy NOT collapsing to 0, ep_return rising.")
else:
    print("no metrics.csv yet — check back once a run has logged some iterations.")


## Recap

You've now seen every layer end-to-end:

- **Engine** — the sole Wolfram touchpoint; `GetSpins` computes gauge-singlet
  bound states + their spins from `PermutationSymmetryOfInvariants`, filtered by
  spin-statistics.
- **Groups / state / actions** — a group graph with dim-ordered rep indices, and a
  pure-Python maskable edit space.
- **Spectrum + cache** — CPT expansion → operator enumeration → an order-invariant
  physics key that makes the 155k-entry shared cache valid across reward changes.
- **Reward** — the `hungarian` trap (score decreasing in `n_pred`; SU(3) worst of
  all groups) and the `f1 + physical` fix (bounded, SU(3) top-tier, adding quarks
  rewarded), all validated offline against the cache.
- **Tokenizer / transformer** — group-invariant featurization feeding a small masked
  actor-critic.
- **PPO** — rollouts vs iterations, GAE, the clipped update, resume-safe entropy
  anneal — and how each quantity shows up in the training logs.

To go deeper: `uv run python tests/test_core.py` exercises the pure-Python logic
(including the reward regression tests), and `scripts/doctor.py` verifies a live
kernel end-to-end.


In [ ]:
# Clean up the kernel session when done (no-op for MockEngine).
try:
    eng.stop(); print("engine stopped.")
except Exception as e:
    print("stop:", e)
